# Démo 1 : un premier neurone qui apprend

Objectif : voir tout le parcours du deep learning en quelques lignes, avant d'en détailler chaque brique : des données, un modèle, une mesure de l'erreur, une boucle qui corrige le modèle, puis une prédiction.

La question : un thermomètre affiche des valeurs dans une unité inconnue. Peut-on apprendre, à partir de quelques mesures, à convertir ses lectures en degrés Celsius ?

**Données** : onze mesures d'un thermomètre dont on ne connaît pas la graduation, avec la température correspondante en degrés Celsius.

**Ce que vous allez voir**

- le parcours complet du deep learning, sur onze mesures : données, modèle, erreur, boucle, prédiction ;
- un neurone qui part de valeurs au hasard et trouve seul la bonne relation ;
- la boucle d'entraînement en cinq lignes, qui servira pendant les deux jours.

**Durée prévue** : 10 min.

*Les sorties enregistrées dans ce notebook viennent d'une exécution dans Google Colab sur GPU A100, avec la graine fixée. Sur un autre GPU ou sur CPU, certaines valeurs peuvent légèrement différer : PyTorch ne garantit pas des résultats identiques d'une plateforme à l'autre.*

## 0. Installation

Cette cellule écrit `requirements-colab.txt` (les versions de la formation, alignées sur le runtime Colab), puis l'installe avec uv. Les paquets déjà présents à la bonne version ne sont pas réinstallés. Hors de Colab, elle n'installe rien : l'environnement uv du projet contient déjà ces versions.

In [ ]:
import importlib.util
from pathlib import Path

Path("requirements-colab.txt").write_text("""torch==2.11.0
torchvision==0.26.0
numpy==2.1.3
matplotlib==3.10.0
pillow==11.3.0
scikit-learn==1.6.1
pandas==2.2.2; python_version < "3.13"
pandas==2.2.3; python_version >= "3.13"
pyarrow>=14
tensorboard==2.21.0
torchviz==0.0.3
onnx==1.23.0
onnxruntime==1.30.0
onnxscript==0.7.2
torchserve==0.12.0
torch-model-archiver==0.12.0
""")

if importlib.util.find_spec("google.colab") is not None:
    %pip install -q --progress-bar off uv
    !uv pip install --system -q -r requirements-colab.txt
else:
    print("Hors Colab : environnement uv du projet, rien à installer.")

Dans Colab, seuls les paquets absents du runtime ou à une autre version sont téléchargés. Si torch ou torchvision venaient à être réinstallés, redémarrez la session (**Exécution > Redémarrer la session**) puis relancez le notebook.

## 1. Vérifier l'environnement

**Première chose à faire dans tout notebook : vérifier la version de PyTorch et la présence d'un GPU.**

Un GPU (carte graphique) fait des milliers de calculs en parallèle ; il accélère fortement
l'entraînement des réseaux. Pour cette première démo, le CPU suffirait, mais on prend l'habitude.

Le code :
- `import torch` : charge PyTorch ;
- `torch.__version__` : la version installée ;
- `torch.cuda.is_available()` : vrai si PyTorch voit un GPU NVIDIA (CUDA est la technologie de calcul de
  NVIDIA) ; `get_device_name(0)` donne le nom du premier GPU.

Dans Colab : menu **Exécution > Modifier le type d'exécution > GPU**, puis exécutez les cellules dans l'ordre.

In [ ]:
import torch

print("torch :", torch.__version__)
print("GPU   :", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "aucun, calcul sur CPU")

PyTorch 2.11 et un GPU NVIDIA A100 : les calculs pourront se faire sur la carte graphique.

**À retenir** : sans GPU, tout le code de la formation tourne aussi, sur le CPU, simplement plus lentement.

## 2. Les données

**Onze mesures : la lecture d'un thermomètre inconnu, et la vraie température en degrés Celsius.**

En deep learning, on ne programme pas la règle de conversion ; on la fait apprendre à partir
d'exemples. Ces onze couples (lecture, température) sont nos exemples.

Le code :
- `torch.tensor([...])` : transforme une liste Python en **tenseur**, le tableau de nombres de PyTorch ;
- `.unsqueeze(1)` : range les valeurs en colonne, une ligne par mesure ; c'est la forme qu'attend un modèle :
  (nombre d'exemples, nombre de valeurs par exemple) ;
- `.shape` : la forme du tenseur.

In [ ]:
lectures = torch.tensor([35.7, 55.9, 58.2, 81.9, 56.3, 48.9, 33.9, 21.8, 48.4, 60.4, 68.4]).unsqueeze(1)
celsius = torch.tensor([0.5, 14.0, 15.0, 28.0, 11.0, 8.0, 3.0, -4.0, 6.0, 13.0, 21.0]).unsqueeze(1)
print(lectures.shape, celsius.shape)

`torch.Size([11, 1])` pour les deux tenseurs : 11 mesures (les lignes), une valeur par mesure (la colonne).
Chaque lecture a sa température, à la même position.

**À retenir** : la première dimension compte les exemples. C'est la convention de PyTorch pour toutes les
données.

## 3. Regarder les données

**Avant de construire un modèle, on regarde toujours les données.**

Le graphique dit quel type de relation chercher, et révèle les valeurs aberrantes. Un modèle ne
voit que des nombres ; nous, nous voyons une forme.

Le code :
- `plt.scatter(x, y)` : un nuage de points, un point par mesure ;
- `plt.xlabel`, `plt.ylabel` : le nom des axes.

Prédisez : les points suivront-ils une droite, une courbe, ou rien de reconnaissable ?

In [ ]:
import matplotlib.pyplot as plt

plt.scatter(lectures, celsius)
plt.xlabel("lecture du thermomètre")
plt.ylabel("degrés Celsius")
plt.show()

Les points s'alignent presque sur une droite, avec un peu de dispersion : les mesures ne sont pas parfaites.
Une relation linéaire semble raisonnable.

**À retenir** : c'est ce graphique qui justifie le choix du modèle de l'étape suivante, une droite.

## 4. Mettre les lectures à l'échelle

**On divise les lectures par 10, pour obtenir des nombres proches de 1.**

L'entraînement corrige les paramètres par petits pas. Si les entrées sont grandes (35, 80), une
petite correction de `w` change beaucoup la prédiction, et l'entraînement devient instable. Des entrées proches
de 1 rendent les pas plus réguliers.

Le code :
- `lectures / 10` : divise chaque valeur du tenseur, sans boucle ;
- `x[:3]` : les trois premières lignes, pour vérifier.

À ne pas oublier : toute nouvelle lecture devra être divisée par 10 de la même façon avant d'être donnée au
modèle.

In [ ]:
x = lectures / 10
print(x[:3])

Les lectures valent maintenant 3.57, 5.59, 5.82... : des nombres proches de 1.

**Piège fréquent** : oublier d'appliquer la même mise à l'échelle aux nouvelles données au moment de prédire.

## 5. Le modèle : un seul neurone

**Le modèle le plus simple possible : un neurone, qui calcule `w * x + b`.**

Les points semblaient suivre une droite ; une droite a deux paramètres, la pente `w` (le **poids**)
et l'ordonnée à l'origine `b` (le **biais**). Tout l'apprentissage consistera à trouver leurs bonnes valeurs.

Le code :
- `import torch.nn as nn` : `nn` contient les briques des réseaux de neurones ;
- `torch.manual_seed(0)` : fixe le hasard, pour obtenir les mêmes résultats à chaque exécution ;
- `nn.Linear(1, 1)` : une couche linéaire, avec **1** valeur en entrée (la lecture) et **1** en sortie (la
  température) ;
- `model.weight.item()`, `model.bias.item()` : les valeurs de `w` et `b`, en nombres Python ;
- `make_dot(model(x), ...)` : calcule les prédictions, puis dessine le graphe des calculs que PyTorch a enregistrés
  pour les obtenir.

In [ ]:
import torch.nn as nn
from torchviz import make_dot

torch.manual_seed(0)
model = nn.Linear(1, 1)
print("w =", model.weight.item(), " b =", model.bias.item())
make_dot(model(x), params=dict(model.named_parameters()))

`w` vaut environ -0.007 et `b` environ 0.536 : des valeurs tirées au hasard. Le neurone ne sait encore rien
du thermomètre.

**Le graphe affiché.** PyTorch a noté chaque calcul fait pour obtenir les prédictions, comme un fil d'Ariane :

- en bleu, les deux paramètres du neurone : `weight` (le poids w, forme `(1, 1)`) et `bias` (le biais b, forme `(1)`).
  Ce sont eux que l'entraînement va ajuster ;
- en gris, les opérations enregistrées : `AddmmBackward0` regroupe « multiplier par le poids, puis ajouter le biais » ;
  `TBackward0` remet le poids dans le bon sens pour la multiplication ;
- en vert, le résultat : `(11, 1)`, une prédiction pour chacune des 11 lectures du thermomètre.

Les flèches descendent, dans le sens du calcul. Pour apprendre, PyTorch remontera ce fil dans l'autre sens, du résultat
vers `weight` et `bias`, pour savoir dans quelle direction corriger chacun d'eux. Les cases `AccumulateGrad` sont
l'endroit où cette correction est déposée.

Pas besoin de retenir les noms des cases : l'idée à garder, c'est que PyTorch construit ce graphe tout seul, à chaque
calcul.

**À retenir** : un modèle neuf a des paramètres aléatoires ; c'est l'entraînement qui leur donne un sens.

## 6. Avant l'entraînement

**Que prédit le neurone avant d'avoir appris quoi que ce soit ?**

C'est notre point de départ. Comparer ses prédictions aux vraies températures montre le chemin à
parcourir.

Le code :
- `with torch.no_grad():` : on demande une simple prédiction, sans préparer d'entraînement (on verra pourquoi
  en démo 3) ;
- `model(x[:3])` : on donne les trois premières lectures au modèle ; il renvoie trois prédictions ;
- `celsius[:3]` : les trois vraies températures, pour comparer.

Prédisez : avec `w` proche de 0, que vaudront les trois prédictions ?

In [ ]:
with torch.no_grad():
    print(model(x[:3]))
print(celsius[:3])

Le neurone prédit environ 0.5 pour les trois mesures, alors que les vraies températures sont 0.5, 14 et
15 degrés : il répond au hasard.

**À retenir** : sans entraînement, un modèle ne vaut rien ; il faut maintenant mesurer son erreur et la réduire.

## 7. L'erreur et le correcteur

**Deux outils pour apprendre : une mesure de l'erreur, et un correcteur qui la fait baisser.**

Pour s'améliorer, le modèle doit savoir à quel point il se trompe (la **loss**, ou fonction de perte)
et comment modifier ses paramètres (l'**optimiseur**).

Le code :
- `nn.MSELoss()` : l'erreur quadratique moyenne ; on fait la différence entre prédiction et vraie valeur, on
  l'élève au carré, et on fait la moyenne sur les exemples ;
- `torch.optim.SGD(model.parameters(), lr=0.01)` : la descente de gradient ; elle reçoit les paramètres à
  corriger, `w` et `b`, et le **learning rate** `lr`, la taille des pas de correction ;
- `.item()` : la valeur de l'erreur, en nombre Python.

In [ ]:
loss_fn = nn.MSELoss()
optimizer = torch.optim.SGD(model.parameters(), lr=0.01)
print("erreur de départ :", loss_fn(model(x), celsius).item())

L'erreur de départ vaut 177.39. C'est une moyenne d'écarts au carré, en degrés au carré : on la lit surtout
en comparant sa valeur avant et après l'entraînement.

**À retenir** : la loss résume en un seul nombre à quel point le modèle se trompe ; l'optimiseur cherche à la
faire baisser.

## 8. La boucle d'entraînement

**Le coeur du deep learning : cinq lignes, répétées 5000 fois.**

Le neurone ne connaît pas la bonne droite. Il part de valeurs au hasard et s'améliore par petites
corrections, en regardant à chaque fois son erreur. Un passage sur toutes les données s'appelle une **époque**.

Le code, ligne par ligne :
- `prediction = model(x)` : le neurone calcule une température pour chacune des 11 lectures ;
- `loss = loss_fn(prediction, celsius)` : on mesure l'erreur par rapport aux vraies températures ;
- `optimizer.zero_grad()` : on efface les corrections calculées au tour précédent (sinon PyTorch les
  additionne, on le verra en démo 3) ;
- `loss.backward()` : PyTorch calcule, pour `w` et pour `b`, dans quel sens les modifier pour réduire l'erreur ;
- `optimizer.step()` : on applique la correction, d'un petit pas réglé par `lr` ;
- `if epoch % 1000 == 0` : on affiche l'erreur toutes les 1000 époques.

Analogie : régler la température de la douche. On essaie, on sent l'écart, on tourne un peu le robinet, et on
recommence.

In [ ]:
for epoch in range(5000):
    prediction = model(x)
    loss = loss_fn(prediction, celsius) # calcul de l'erreur
    optimizer.zero_grad()
    loss.backward() # calcul du gradient
    optimizer.step() # mise à jour des poids
    if epoch % 1000 == 0:
        print(f"époque {epoch:4d}  erreur {loss.item():.2f}")

L'erreur tombe de 177.39 à 3.90 après 1000 époques, puis se stabilise à 2.93 : le neurone ne peut pas faire
mieux, car les mesures elles-mêmes sont un peu imprécises.

**À retenir** : ces cinq lignes sont la boucle de tout entraînement ; cet après-midi, elles entraîneront un
réseau d'images, sans changement.

**Essayez** : remplacez `lr=0.01` par `lr=0.1`, puis par `lr=0.0001`, relancez depuis l'étape 5, et
observez l'erreur.

## 9. Le résultat

**La droite apprise par le neurone, tracée sur les mesures.**

L'erreur est un nombre ; le graphique montre ce qu'elle signifie. Un bon modèle passe au milieu des
points.

Le code :
- `with torch.no_grad():` : on trace des prédictions, sans entraîner ;
- `plt.scatter(...)` : les mesures ;
- `plt.plot(lectures, model(x), ...)` : les prédictions du neurone, reliées par une ligne rouge ;
- `plt.legend()` : la légende.

In [ ]:
with torch.no_grad():
    plt.scatter(lectures, celsius, label="mesures")
    plt.plot(lectures, model(x), color="red", label="neurone")
plt.xlabel("lecture du thermomètre")
plt.ylabel("degrés Celsius")
plt.legend()
plt.show()

La droite rouge passe au milieu des mesures : le neurone a trouvé la relation linéaire.

**À retenir** : le graphique confirme ce que disait l'erreur ; les deux se complètent.

## 10. Prédire

**Le thermomètre affiche 65 : quelle température en degrés Celsius ?**

C'est le but de tout modèle, prédire pour une valeur qu'il n'a jamais vue.

Le code :
- `torch.tensor([[65.0]])` : une nouvelle lecture, au même format que les données : une ligne, une colonne ;
- `/ 10` : la même mise à l'échelle qu'à l'entraînement ;
- `model(...)` sous `torch.no_grad()` : la prédiction.

Prédisez : d'après le graphique précédent, autour de quelle valeur ?

In [ ]:
with torch.no_grad():
    print(model(torch.tensor([[65.0]]) / 10))

Une lecture de 65 correspond à environ 17.6 degrés Celsius.

**Essayez** : prédisez pour une lecture de 32, puis de 212. Que remarquez-vous ?

## 11. Ce que le neurone a appris

**Les deux paramètres appris résument toute la relation.**

Un neurone seul est lisible ; on peut interpréter ses paramètres. Ce ne sera plus le cas avec des
réseaux de millions de paramètres, mais le principe reste le même : l'apprentissage fixe des nombres.

Le code :
- `round(..., 3)` : arrondit à trois décimales pour l'affichage.

In [ ]:
print("w =", round(model.weight.item(), 3), " b =", round(model.bias.item(), 3))

`w = 5.367` et `b = -17.301` : la température vaut `w` fois la lecture mise à l'échelle, plus `b`. Ce sont
presque les coefficients de la conversion des degrés Fahrenheit en degrés Celsius : le thermomètre était
gradué en Fahrenheit. Le neurone a retrouvé la formule à partir de onze mesures, sans qu'on la lui donne.

**À retenir** : apprendre, c'est trouver les valeurs des paramètres à partir d'exemples.

## À retenir de cette démo

- Un modèle a des **paramètres** (ici `w` et `b`) ; l'entraînement les ajuste pour réduire une **erreur** (la loss).
- La boucle d'entraînement tient en cinq lignes : prédire, mesurer l'erreur, `zero_grad()`, `backward()`, `step()`.
- Un modèle entraîné sert ensuite à prédire, sous `torch.no_grad()`.